# 02 · Очистка данных

Ноутбук очищает данные перед объединением и анализом.

- **Часть 1:** погода и модельный PM2.5 (CAMS) из Open-Meteo.
- **Часть 2:** PM2.5 станции OpenAQ (шаги 12–23).

Порядок шагов — по Лекции 4: инспекция → имена → типы → дубликаты → пропуски → невозможные значения → выбросы → итоговые проверки.

Сырые файлы из `data/raw/` только читаем. Результат записываем в `data/processed/`. Запускать сверху вниз.

# Часть 1. Погода и CAMS

## Шаг 1. Импорты и пути

Подключаем pandas и наши модули `src/clean.py` и `src/collect.py`, задаём пути к файлам и списки колонок.

In [1]:
import sys

import pandas as pd

sys.path.append("..")
from src import clean, collect

WEATHER_RAW_PATH = "../data/raw/weather.csv"
CAMS_RAW_PATH = "../data/raw/cams_aq.csv"
WEATHER_CLEAN_PATH = "../data/processed/weather_clean.csv"
CAMS_CLEAN_PATH = "../data/processed/cams_clean.csv"

WEATHER_COLUMNS = list(collect.WEATHER_COLUMNS.values())
CAMS_COLUMNS = list(collect.CAMS_COLUMNS.values())

## Шаг 2. Инспекция

По Лекции 4 сначала смотрим на данные, потом исправляем. Подробный осмотр сделан в `01_collect` (шаги 12–15), здесь — короткая сводка: размер, типы, пропуски.

In [2]:
weather_raw = pd.read_csv(WEATHER_RAW_PATH)
cams_raw = pd.read_csv(CAMS_RAW_PATH)

for name, table in [("weather", weather_raw), ("cams", cams_raw)]:
    print(name, table.shape)
    print(table.dtypes)
    print(table.isna().sum())
    print()

weather_raw.describe()

weather (49104, 7)
time_utc          str
temp_c        float64
humidity        int64
precip        float64
pressure      float64
wind_speed    float64
wind_dir        int64
dtype: object
time_utc      0
temp_c        0
humidity      0
precip        0
pressure      0
wind_speed    0
wind_dir      0
dtype: int64

cams (28776, 2)
time_utc         str
cams_pm25    float64
dtype: object
time_utc     0
cams_pm25    0
dtype: int64



,temp_c,humidity,precip,pressure,wind_speed,wind_dir
count,49104.000000,49104.000000,49104.000000,49104.000000,49104.000000,49104.00000
mean,11.873153,55.073884,0.059883,1018.549723,1.563412,181.14897
std,11.913288,21.377138,0.323194,9.978545,0.910536,106.37092
min,-28.400000,7.000000,0.000000,994.600000,0.000000,1.00000
25%,2.400000,38.000000,0.000000,1010.800000,0.920000,101.00000
50%,13.000000,53.000000,0.000000,1017.500000,1.430000,169.00000
75%,21.300000,72.000000,0.000000,1025.400000,2.020000,270.00000
max,39.100000,100.000000,10.600000,1058.900000,10.740000,360.00000


## Шаг 3. Имена колонок

Имена проекта в snake_case заданы при выгрузке, значения не менялись. Модельный PM2.5 называется `cams_pm25`. Проверяем, что имена и порядок ожидаемые.

In [3]:
clean.check_columns(weather_raw, WEATHER_COLUMNS)
clean.check_columns(cams_raw, CAMS_COLUMNS)
print(list(weather_raw.columns))
print(list(cams_raw.columns))

['time_utc', 'temp_c', 'humidity', 'precip', 'pressure', 'wind_speed', 'wind_dir']
['time_utc', 'cams_pm25']


## Шаг 4. Типы: время в UTC

Сырые метки времени — строки без `Z`, поэтому `utc=True` и `format=` задаём явно. `errors="coerce"` превращает битые метки в NaT, а функция сразу считает их и падает с понятным сообщением. Числовые колонки уже числа.

In [4]:
weather = weather_raw.copy()
weather["time_utc"] = clean.parse_utc_times(weather_raw["time_utc"], collect.METEO_TIME_FORMAT)
cams = cams_raw.copy()
cams["time_utc"] = clean.parse_utc_times(cams_raw["time_utc"], collect.METEO_TIME_FORMAT)

for name, table in [("weather", weather), ("cams", cams)]:
    print(name, table["time_utc"].dtype, "NaT:", table["time_utc"].isna().sum())
    print("первая:", table["time_utc"].iloc[0], "последняя:", table["time_utc"].iloc[-1])

weather datetime64[us, UTC] NaT: 0
первая: 2020-04-09 00:00:00+00:00 последняя: 2025-11-14 23:00:00+00:00
cams datetime64[us, UTC] NaT: 0
первая: 2022-08-04 00:00:00+00:00 последняя: 2025-11-14 23:00:00+00:00


## Шаг 5. Местное время Алматы

Анализ ведём по `Asia/Almaty`. 1 марта 2024 пояс сменился с UTC+6 на UTC+5, поэтому смотрим стык: какие местные метки получаются из UTC-часов вокруг перехода.

In [5]:
junction = weather[
    (weather["time_utc"] >= pd.Timestamp("2024-02-29 16:00", tz="UTC"))
    & (weather["time_utc"] <= pd.Timestamp("2024-02-29 19:00", tz="UTC"))
]
pd.DataFrame({
    "time_utc": junction["time_utc"],
    "time_almaty": clean.to_almaty_time(junction["time_utc"]),
})

,time_utc,time_almaty
34120,2024-02-29 16:00:00+00:00,2024-02-29 22:00:00+06:00
34121,2024-02-29 17:00:00+00:00,2024-02-29 23:00:00+06:00
34122,2024-02-29 18:00:00+00:00,2024-02-29 23:00:00+05:00
34123,2024-02-29 19:00:00+00:00,2024-03-01 00:00:00+05:00


Считаем, сколько раз повторяется местная метка, если отбросить смещение: это показывает, почему местное время нельзя брать ключом. Заодно считаем, сколько часов длится 29.02.2024 по местному времени.

In [6]:
for name, table in [("weather", weather), ("cams", cams)]:
    local_naive = clean.to_almaty_time(table["time_utc"]).dt.tz_localize(None)
    print(name, "повторов местного времени без смещения:", local_naive.duplicated().sum())

local_day = clean.to_almaty_time(weather["time_utc"]).dt.date
hours_in_day = (local_day == pd.Timestamp("2024-02-29").date()).sum()
print("часов 29.02.2024 по местному времени:", hours_in_day)

weather повторов местного времени без смещения: 1
cams повторов местного времени без смещения: 1
часов 29.02.2024 по местному времени: 25


Местная метка 23:00 29.02.2024 встречается дважды: один раз со смещением +06:00, второй раз +05:00. Поэтому ключом для объединения оставляем `time_utc` — он не прыгает. В файл местное время не пишем, а `clean.to_almaty_time` применяем при построении признаков (задача 3.1).

## Шаг 6. Дубликаты

Опасны дубликаты ключа: повторяющийся час сломает объединение. Считаем повторы `time_utc`.

In [7]:
print("weather дублей time_utc:", weather["time_utc"].duplicated().sum())
print("cams дублей time_utc:", cams["time_utc"].duplicated().sum())

weather дублей time_utc: 0
cams дублей time_utc: 0


## Шаг 7. Пропуски

Проверяем, что сетка часов полная (между первой и последней меткой нет «дыр») и что пустых значений нет. Если пропусков 0, интерполяция не нужна. Начало CAMS (2022-08-04) — это период источника, а не пропуск.

In [8]:
for name, table in [("weather", weather), ("cams", cams)]:
    first = table["time_utc"].min()
    last = table["time_utc"].max()
    expected_hours = int((last - first) / pd.Timedelta(hours=1)) + 1
    print(name, "строк:", len(table), "ожидаемо часов:", expected_hours)
    print(table.isna().sum())

weather строк: 49104 ожидаемо часов: 49104
time_utc      0
temp_c        0
humidity      0
precip        0
pressure      0
wind_speed    0
wind_dir      0
dtype: int64
cams строк: 28776 ожидаемо часов: 28776
time_utc     0
cams_pm25    0
dtype: int64


## Шаг 8. Невозможные значения

Границы `clean.VALID_RANGES` — физически возможные значения. Всё вне них — ошибка данных, а не выброс. Считаем нарушения по каждой колонке и требуем 0.

In [9]:
weather_bad = clean.count_out_of_range(weather, clean.VALID_RANGES)
cams_bad = clean.count_out_of_range(cams, clean.VALID_RANGES)
print("weather:", weather_bad)
print("cams:", cams_bad)
total_bad = sum(weather_bad.values()) + sum(cams_bad.values())
assert total_bad == 0, f"значения вне допустимых диапазонов: {total_bad}"

weather: {'temp_c': 0, 'humidity': 0, 'precip': 0, 'pressure': 0, 'wind_speed': 0, 'wind_dir': 0}
cams: {'cams_pm25': 0}


## Шаг 9. Выбросы по IQR

По Лекции 4 выбросы по IQR — «прожектор, а не приговор»: мы только считаем и смотрим, ничего не удаляем (пики смога и сильный ветер — реальные явления). Для `precip` IQR не считаем: не меньше 75 % часов без осадков, поэтому Q1 = Q3 = 0 и IQR = 0, и любой дождь стал бы «выбросом» (покажем долю часов без осадков и квартили). Для `wind_dir` не считаем: это угол по кругу, 1° и 360° почти одно направление.

In [10]:
IQR_COLUMNS = ["temp_c", "humidity", "pressure", "wind_speed"]
for column in IQR_COLUMNS:
    mask = clean.iqr_outlier_mask(weather[column])
    print(column, "выбросов:", mask.sum(), "доля, %:", round(mask.mean() * 100, 3))

cams_mask = clean.iqr_outlier_mask(cams["cams_pm25"])
print("cams_pm25", "выбросов:", cams_mask.sum(), "доля, %:", round(cams_mask.mean() * 100, 3))

print("precip доля часов без осадков:", round((weather["precip"] == 0).mean(), 3))
print("precip Q1:", weather["precip"].quantile(0.25), "Q3:", weather["precip"].quantile(0.75))

temp_c выбросов: 13 доля, %: 0.026
humidity выбросов: 0 доля, %: 0.0
pressure выбросов: 213 доля, %: 0.434
wind_speed выбросов: 1415 доля, %: 2.882
cams_pm25 выбросов: 1520 доля, %: 5.282
precip доля часов без осадков: 0.911
precip Q1: 0.0 Q3: 0.0


Посмотрим 5 самых больших значений модельного PM2.5 вместе с местным временем: это зимние пики модели CAMS (6 января 2023, вечер по местному времени), а не ошибки данных — их оставляем.

In [11]:
top_cams = cams.nlargest(5, "cams_pm25").copy()
top_cams["time_almaty"] = clean.to_almaty_time(top_cams["time_utc"])
print(top_cams)

                      time_utc  cams_pm25               time_almaty
3731 2023-01-06 11:00:00+00:00      153.7 2023-01-06 17:00:00+06:00
3732 2023-01-06 12:00:00+00:00      147.1 2023-01-06 18:00:00+06:00
3730 2023-01-06 10:00:00+00:00      146.2 2023-01-06 16:00:00+06:00
3729 2023-01-06 09:00:00+00:00      132.7 2023-01-06 15:00:00+06:00
3733 2023-01-06 13:00:00+00:00      129.9 2023-01-06 19:00:00+06:00


## Шаг 10. Итоговые проверки

Контракт для объединения: один час UTC — одна строка, время по возрастанию с шагом 1 час, без пропусков и невозможных значений. Любое нарушение остановит ноутбук.

In [12]:
clean.check_clean_hourly(weather, WEATHER_COLUMNS)
clean.check_clean_hourly(cams, CAMS_COLUMNS)
print("проверки пройдены")

проверки пройдены


## Шаг 11. Сохраняем очищенные таблицы

Сырые файлы не трогаем. Очищенные пишем в `data/processed/` с явным форматом времени (смещение `+0000` говорит, что это UTC). Эти файлы перезаписываются при каждом запуске — это производные данные. Читаем их обратно и проверяем ещё раз.

In [13]:
clean.save_clean_csv(weather, WEATHER_CLEAN_PATH)
clean.save_clean_csv(cams, CAMS_CLEAN_PATH)

weather_check = clean.read_clean_csv(WEATHER_CLEAN_PATH)
cams_check = clean.read_clean_csv(CAMS_CLEAN_PATH)
clean.check_clean_hourly(weather_check, WEATHER_COLUMNS)
clean.check_clean_hourly(cams_check, CAMS_COLUMNS)
print("weather_clean:", weather_check.shape)
print("cams_clean:", cams_check.shape)
print(weather_check.head(3))
print(cams_check.head(3))

weather_clean: (49104, 7)
cams_clean: (28776, 2)
                   time_utc  temp_c  humidity  precip  pressure  wind_speed  \
0 2020-04-09 00:00:00+00:00     2.4        94     0.5    1026.8        2.33   
1 2020-04-09 01:00:00+00:00     2.2        99     0.5    1027.5        0.72   
2 2020-04-09 02:00:00+00:00     2.5        94     0.3    1028.1        1.14   

   wind_dir  
0       245  
1       326  
2       232  
                   time_utc  cams_pm25
0 2022-08-04 00:00:00+00:00       15.1
1 2022-08-04 01:00:00+00:00       15.9
2 2022-08-04 02:00:00+00:00       15.4


## Итог части 1

- **Что нашли:** пропусков, дублей и значений вне допустимых диапазонов нет (см. вывод шагов 7, 8 и 6); выбросы по IQR посчитаны и оставлены (шаг 9); местное время 23:00 29.02.2024 встречается дважды (шаг 5).
- **Что изменили:** только тип времени — строки стали датой с поясом UTC. Значения не менялись.
- **Что записали:** `data/processed/weather_clean.csv` и `data/processed/cams_clean.csv`.
- **Для задачи 2.2:** функции `parse_utc_times`, `to_almaty_time`, `count_out_of_range`, `iqr_outlier_mask`, `save_clean_csv` и `read_clean_csv` готовы.
- **Для задачи 3.1:** ключ объединения — `time_utc`; местное время считаем через `clean.to_almaty_time`; день 29.02.2024 по местному времени длится столько часов, сколько показал вывод шага 5.

# Часть 2. PM2.5 станции

Источник — станция OpenAQ 8876 в Алматы, почасовые средние PM2.5 в µg/m³. Это главный ряд проекта: его мы объясняем и прогнозируем.

Порядок шагов — по Лекции 4: инспекция и имена → типы → дубликаты → невозможные значения → флаги поставщика → полная сетка часов (видим реальные дыры) → длины дыр → нули и застывший датчик → выбросы и пики → интерполяция коротких дыр с флагами → итоговые проверки → запись.

Интерполяцию делаем **последней**: сначала убираем всё, что превращается в пропуск (флаги, застывший датчик), и только потом заполняем короткие дыры. Иначе мы бы заполнили значение, а потом его выбросили.

Результат — `data/processed/pm25_clean.csv` на полной сетке часов станции.

## Шаг 12. Инспекция и имена

Читаем сырой файл и смотрим на него, ничего не меняя. Колонки уже в snake_case, проверяем, что они совпадают с тем, что записал сбор данных (`collect.RAW_COLUMNS`).

In [14]:
PM25_RAW_PATH = "../data/raw/openaq_pm25.csv"
PM25_CLEAN_PATH = "../data/processed/pm25_clean.csv"

pm25_raw = pd.read_csv(PM25_RAW_PATH)
clean.check_columns(pm25_raw, list(collect.RAW_COLUMNS.values()))

print("shape:", pm25_raw.shape)
print(pm25_raw.dtypes)
print(pm25_raw.isna().sum())
print(pm25_raw.describe())
print(pm25_raw.head())

shape: (37823, 4)
datetime_from_utc        str
datetime_to_utc          str
pm25                 float64
has_flags               bool
dtype: object
datetime_from_utc     0
datetime_to_utc       0
pm25                 66
has_flags             0
dtype: int64
               pm25
count  37757.000000
mean      29.547236
std       37.862882
min        0.000000
25%        9.000000
50%       15.000000
75%       31.000000
max      421.000000
      datetime_from_utc       datetime_to_utc  pm25  has_flags
0  2020-04-09T16:00:00Z  2020-04-09T17:00:00Z   6.0      False
1  2020-04-09T17:00:00Z  2020-04-09T18:00:00Z  10.0      False
2  2020-04-09T18:00:00Z  2020-04-09T19:00:00Z   8.0      False
3  2020-04-09T19:00:00Z  2020-04-09T20:00:00Z   6.0      False
4  2020-04-09T20:00:00Z  2020-04-09T21:00:00Z   4.0      False


## Шаг 13. Типы: время в UTC и ключ часа

В сыром файле две метки: начало и конец часа измерения. Проверяем, что каждый интервал ровно 1 час. Ключом `time_utc` берём **начало часа**: так подписывают часы сети AirNow, суточное среднее по местным суткам берёт ровно часы, начавшиеся в эти сутки, и признак «час суток» читается естественно (решение ADR-012).

Вторая метка новой информации не несёт, в таблицу её не переносим. Колонку `has_flags` пока оставляем — она нужна в шаге 16.

In [15]:
time_from = clean.parse_utc_times(pm25_raw["datetime_from_utc"], collect.TIME_FORMAT)
time_to = clean.parse_utc_times(pm25_raw["datetime_to_utc"], collect.TIME_FORMAT)
assert ((time_to - time_from) == pd.Timedelta(hours=1)).all(), "интервал не равен 1 часу"

pm25 = pd.DataFrame(
    {"time_utc": time_from, "pm25": pm25_raw["pm25"], "has_flags": pm25_raw["has_flags"]}
)
print("dtype времени:", pm25["time_utc"].dtype)
print("первая метка:", pm25["time_utc"].iloc[0])
print("последняя метка:", pm25["time_utc"].iloc[-1])

dtype времени: datetime64[us, UTC]
первая метка: 2020-04-09 16:00:00+00:00
последняя метка: 2025-11-14 10:00:00+00:00


## Шаг 14. Дубликаты

Один час должен встречаться один раз. Повтор означал бы ошибку сбора или объединения страниц API.

In [16]:
duplicates = pm25["time_utc"].duplicated().sum()
assert duplicates == 0, "есть повторяющиеся часы"
print("повторяющихся часов:", duplicates)

повторяющихся часов: 0


## Шаг 15. Невозможные значения

PM2.5 не может быть отрицательным, а значения выше 1000 µg/m³ — ошибка прибора (допустимый диапазон в `clean.VALID_RANGES`). Пустое значение нарушением не считаем: пропуски разбираем отдельно.

In [17]:
print("допустимый диапазон:", clean.VALID_RANGES["pm25"])
out_of_range = clean.count_out_of_range(pm25, clean.VALID_RANGES)
print("нарушений:", out_of_range)
assert sum(out_of_range.values()) == 0, "есть значения вне диапазона"
print("min:", pm25["pm25"].min(), "max:", pm25["pm25"].max())

допустимый диапазон: (0, 1000)
нарушений: {'pm25': 0}
min: 0.0 max: 421.0


## Шаг 16. Флаги поставщика

Колонка `has_flags` — пометка самого поставщика данных, что значение сомнительное. Решение: такие значения превращаем в пропуск. Причины: все пустые значения оказываются среди флагованных, то есть флаг сопровождает сбои; а непустых флагованных мало, потеря ничтожна.

Проверяем, что флаг не отмечает реальные пики смога: считаем, сколько флагованных значений выше 99-го перцентиля всех значений. Если их 10 и больше — флаг может прятать настоящие пики, и решение надо пересматривать (`assert` остановит ноутбук). Делаем это **до** построения сетки, пока колонка `has_flags` ещё в таблице.

In [18]:
flagged = pm25["has_flags"]
print("строк с флагом:", flagged.sum())
print("  из них с пустым pm25:", (flagged & pm25["pm25"].isna()).sum())
print("  из них с непустым pm25:", (flagged & pm25["pm25"].notna()).sum())

flagged_values = pm25.loc[flagged, "pm25"].dropna()
print("значения с флагом:")
print(flagged_values.describe())
print("все значения:")
print(pm25["pm25"].describe())

p99 = pm25["pm25"].quantile(0.99)
high_flagged = (flagged_values >= p99).sum()
print("99-й перцентиль всех значений:", p99)
print("флагованных значений не ниже p99:", high_flagged)
assert high_flagged < 10, "флаги отмечают высокие значения: пересмотреть решение"

empty_before = pm25["pm25"].isna().sum()
pm25["pm25"] = pm25["pm25"].mask(pm25["has_flags"])
pm25 = pm25.drop(columns="has_flags")
print("пустых pm25 до:", empty_before, "после:", pm25["pm25"].isna().sum())

строк с флагом: 113
  из них с пустым pm25: 66
  из них с непустым pm25: 47
значения с флагом:
count    47.000000
mean      2.361702
std       3.299707
min       0.000000
25%       0.000000
50%       1.000000
75%       3.000000
max      17.000000
Name: pm25, dtype: float64
все значения:
count    37757.000000
mean        29.547236
std         37.862882
min          0.000000
25%          9.000000
50%         15.000000
75%         31.000000
max        421.000000
Name: pm25, dtype: float64
99-й перцентиль всех значений: 189.0
флагованных значений не ниже p99: 0
пустых pm25 до: 66 после: 113


**Вывод.** Флагованных значений не ниже 99-го перцентиля нет (см. вывод выше), поэтому флаги не прячут пики смога. Значения с флагом заменены пропусками, колонка `has_flags` больше не нужна.

## Шаг 17. Полная сетка часов

В сыром файле нет строк для часов, когда станция ничего не передала: пропущенный час просто отсутствует. Метод `asfreq("h")` (внутри `clean.to_hourly_grid`) добавляет такие часы строкой с пустым `pm25`, и пропуски становятся видимыми.

Границы сетки — первая и последняя метка станции, а не период погоды: погода шире, и расширять PM2.5 на часы, когда станция ещё или уже не работала, значит придумать пропуски, которых нет.

In [19]:
grid = clean.to_hourly_grid(pm25)
EXPECTED_HOURS = 49_075  # часов от первой до последней метки станции (проверено при пробе данных)
assert len(grid) == EXPECTED_HOURS, "неожиданное число часов в сетке"

print("часов в сетке:", len(grid))
print("первая метка:", grid["time_utc"].iloc[0])
print("последняя метка:", grid["time_utc"].iloc[-1])

empty_hours = grid["pm25"].isna().sum()
print("пустых часов:", empty_hours, f"({empty_hours / len(grid):.2%})")

local_year = clean.to_almaty_time(grid["time_utc"]).dt.year
print("пустых часов по местным годам:")
print(grid["pm25"].isna().groupby(local_year).agg(["sum", "size"]))

часов в сетке: 49075
первая метка: 2020-04-09 16:00:00+00:00
последняя метка: 2025-11-14 10:00:00+00:00
пустых часов: 11365 (23.16%)
пустых часов по местным годам:
           sum  size
time_utc            
2020      1700  6386
2021      3660  8760
2022      4046  8760
2023      1431  8760
2024       482  8785
2025        46  7624


## Шаг 18. Длины дыр

Правило Лекции 4 — короткий пробел во временном ряду можно заполнить, длинный нельзя. Чтобы его применить, надо знать длину каждой дыры. `clean.gap_sizes` даёт одну строку на дыру, `pd.cut` раскладывает дыры по интервалам длины.

In [20]:
sizes = clean.gap_sizes(grid["pm25"])
print("дыр всего:", len(sizes), "часов в них:", sizes.sum())

bins = pd.cut(sizes, bins=[0, 1, 2, 3, 6, 24, 168, float("inf")])
print(sizes.groupby(bins, observed=False).agg(["size", "sum"]))

short = sizes[sizes <= clean.MAX_INTERPOLATION_HOURS]
print("дыр не длиннее 3 ч:", len(short), "часов в них:", short.sum())

lengths = clean.gap_lengths(grid["pm25"])
longest = grid.loc[lengths == sizes.max(), "time_utc"]
print("самая длинная дыра:", sizes.max(), "ч, с", longest.min(), "по", longest.max())

is_empty = grid["pm25"].isna()
gap_number = clean.run_ids(is_empty)
gap_table = grid[is_empty].groupby(gap_number[is_empty]).agg(
    start=("time_utc", "first"), end=("time_utc", "last"), hours=("pm25", "size")
)
print("5 самых длинных дыр:")
print(gap_table.nlargest(5, "hours").reset_index(drop=True))

дыр всего: 1130 часов в них: 11365
               size   sum
pm25                     
(0.0, 1.0]      719   719
(1.0, 2.0]      153   306
(2.0, 3.0]       41   123
(3.0, 6.0]       53   246
(6.0, 24.0]     112  1780
(24.0, 168.0]    43  2210
(168.0, inf]      9  5981
дыр не длиннее 3 ч: 913 часов в них: 1148
самая длинная дыра: 3097 ч, с 2022-04-30 17:00:00+00:00 по 2022-09-06 17:00:00+00:00
5 самых длинных дыр:
                      start                       end  hours
0 2022-04-30 17:00:00+00:00 2022-09-06 17:00:00+00:00   3097
1 2023-02-16 01:00:00+00:00 2023-03-19 00:00:00+00:00    744
2 2021-10-10 02:00:00+00:00 2021-11-05 06:00:00+00:00    629
3 2021-06-22 12:00:00+00:00 2021-07-09 04:00:00+00:00    401
4 2023-02-02 01:00:00+00:00 2023-02-14 00:00:00+00:00    288


**Вывод.** Большинство дыр короткие (1–3 часа), но основная масса пустых часов приходится на несколько длинных дыр: самые длинные — в 2021–2023 годах (см. список выше). По доле пустых часов заметно выделяются 2021 и 2022 годы, но и в 2020 году пустых часов много (см. таблицу по годам в шаге 17). Длинные дыры мы не заполняем — это ограничение для EDA и модели (задачи 4.1 и 6.1).

## Шаг 19. Нули и застывший датчик

Сбой датчика мы определяем так: одно и то же значение `pm25` держится `clean.STUCK_MIN_HOURS` часов подряд и дольше. Берём 6 часов: прибор выдаёт целые µg/m³, поэтому 2–4 одинаковых часа подряд бывают и у исправного датчика (особенно ночью при малых значениях), а шесть подряд для меняющегося воздуха маловероятны. Пропавший час прерывает серию, иначе «подряд» склеило бы часы через дыру. Если под правило попадёт больше 1 % значений, это уже свойство данных, а не сбой — проверяем `assert`.

Нули: 0 µg/m³ лежит в допустимом диапазоне. Нули внутри застывших серий уйдут по правилу выше, одиночные оставляем. Чтобы понять, не признак ли это провала датчика, смотрим на соседей: если рядом чистый воздух (среднее соседних часов ниже 20 µg/m³), ноль правдоподобен.

In [21]:
stuck = clean.stuck_value_mask(grid["pm25"])
non_empty = grid["pm25"].notna().sum()
stuck_share = stuck.sum() / non_empty
print("застывших часов:", stuck.sum(), f"({stuck_share:.3%} непустых значений)")
assert stuck_share <= 0.01, "под правило попало больше 1 % значений"

print("значения в застывших сериях:")
print(grid.loc[stuck, "pm25"].value_counts())

run_number = clean.run_ids(grid["pm25"])
stuck_runs = grid[stuck].groupby(run_number[stuck]).agg(
    start=("time_utc", "first"), value=("pm25", "first"), hours=("pm25", "size")
)
print("застывших серий:", len(stuck_runs))
print(stuck_runs.sort_values("hours", ascending=False).head(10))

is_zero = grid["pm25"] == 0
neighbors_mean = (grid["pm25"].shift(1) + grid["pm25"].shift(-1)) / 2
print("нулей всего:", is_zero.sum(), "из них в застывших сериях:", (is_zero & stuck).sum())
print("среднее соседних часов у остальных нулей:")
print(neighbors_mean[is_zero & ~stuck].describe())
print("медиана:", neighbors_mean[is_zero & ~stuck].median())

застывших часов: 18 (0.048% непустых значений)
значения в застывших сериях:
pm25
6.0     12
13.0     6
Name: count, dtype: int64
застывших серий: 3
                          start  value  hours
pm25                                         
25591 2023-05-03 18:00:00+00:00    6.0      6
33435 2024-04-30 13:00:00+00:00   13.0      6
38372 2024-12-18 19:00:00+00:00    6.0      6
нулей всего: 77 из них в застывших сериях: 0
среднее соседних часов у остальных нулей:
count    56.000000
mean      2.125000
std       1.763906
min       0.000000
25%       1.000000
50%       1.500000
75%       2.500000
max      11.500000
Name: pm25, dtype: float64
медиана: 1.5


Применяем решение: застывшие значения заменяем пропусками. Вывод по нулям — в ячейке ниже.

In [22]:
empty_before = grid["pm25"].isna().sum()
grid["pm25"] = grid["pm25"].mask(stuck)
print("пустых часов до:", empty_before, "после:", grid["pm25"].isna().sum())

пустых часов до: 11365 после: 11383


**Вывод по нулям.** Медиана среднего соседних часов у одиночных нулей ниже порога 20 µg/m³ (см. вывод выше): нули стоят среди чистого воздуха, это правдоподобные измерения. Нули оставляем.

## Шаг 20. Выбросы по IQR и пики смога

Метод IQR (L4) — прожектор, а не приговор: он показывает необычные значения, но не решает, что с ними делать. Пики смога — само изучаемое явление, поэтому мы ничего не удаляем, а только считаем выбросы, смотрим, когда они случаются, и проверяем самые высокие значения вместе с соседними часами.

In [23]:
outliers = clean.iqr_outlier_mask(grid["pm25"])
q1 = grid["pm25"].quantile(0.25)
q3 = grid["pm25"].quantile(0.75)
upper = q3 + 1.5 * (q3 - q1)
print("Q1:", q1, "Q3:", q3, "верхняя граница:", upper)
print("выбросов:", outliers.sum(), f"({outliers.sum() / grid['pm25'].notna().sum():.2%} непустых)")

local_month = clean.to_almaty_time(grid["time_utc"]).dt.month
print("выбросов по местным месяцам:")
print(outliers.groupby(local_month).sum())

top5 = grid["pm25"].nlargest(5).index
top_table = pd.DataFrame({
    "local_time": clean.to_almaty_time(grid["time_utc"]).loc[top5],
    "before": grid["pm25"].shift(1).loc[top5],
    "pm25": grid["pm25"].loc[top5],
    "after": grid["pm25"].shift(-1).loc[top5],
})
print("топ-5 значений с соседними часами:")
print(top_table)

Q1: 9.0 Q3: 31.0 верхняя граница: 64.0
выбросов: 4582 (12.16% непустых)
выбросов по местным месяцам:
time_utc
1     1256
2     1054
3      151
4       10
5        0
6        0
7        4
8        0
9        4
10      62
11     699
12    1342
Name: pm25, dtype: int64
топ-5 значений с соседними часами:
                     local_time  before   pm25  after
6836  2021-01-19 18:00:00+06:00   372.0  421.0    NaN
6601  2021-01-09 23:00:00+06:00   339.0  396.0  391.0
6602  2021-01-10 00:00:00+06:00   396.0  391.0  357.0
23802 2022-12-27 16:00:00+06:00   316.0  375.0  121.0
6835  2021-01-19 17:00:00+06:00   138.0  372.0  421.0


**Вывод.** Выбросы сосредоточены в отопительном сезоне (ноябрь – февраль, см. таблицу по месяцам), летом их почти нет. Самые высокие значения не одиночные: соседние часы тоже высокие, то есть это настоящие эпизоды смога. Выбросы и пики не удаляем.

## Шаг 21. Короткие дыры: интерполяция и флаги

Короткий пробел во временном ряду заполняем линейной интерполяцией (`interpolate`, L4), длинный не выдумываем: оставляем пустым и помечаем флагом. Заполняем только дыры целиком не длиннее 3 часов. Простой `interpolate(limit=3)` так не умеет: он заполнил бы первые 3 часа и у длинной дыры. Покажем это числом.

In [24]:
filled = clean.interpolate_short_gaps(grid["pm25"])
pm25_clean = pd.DataFrame({
    "time_utc": grid["time_utc"],
    "pm25": filled,
    "pm25_missing": filled.isna(),
    "pm25_interpolated": grid["pm25"].isna() & filled.notna(),
})

hours_before = grid["pm25"].isna().sum()
filled_hours = pm25_clean["pm25_interpolated"].sum()
print("пустых часов до:", hours_before)
print("заполнено часов:", filled_hours)
print("осталось пустых часов:", pm25_clean["pm25_missing"].sum(),
      f"({pm25_clean['pm25_missing'].mean():.2%})")

# ловушка limit=3: сколько часов внутри длинных дыр заполнил бы простой interpolate
naive = grid["pm25"].interpolate(limit=3, limit_area="inside")
in_long_gap = clean.gap_lengths(grid["pm25"]) > 3
trap_hours = (grid["pm25"].isna() & naive.notna() & in_long_gap).sum()
print("часов внутри длинных дыр, которые заполнил бы interpolate(limit=3):", trap_hours)

пустых часов до: 11383
заполнено часов: 1148
осталось пустых часов: 10235 (20.86%)
часов внутри длинных дыр, которые заполнил бы interpolate(limit=3): 660


## Шаг 22. Итоговые проверки

Контракт таблицы для задачи 3.1: колонки, порядок, время по сетке без дублей, флаги типа bool, `pm25_missing` совпадает с пустыми `pm25`, диапазон значений, а оставшиеся дыры длиннее 3 часов.

In [25]:
clean.check_clean_pm25(pm25_clean)
print("проверки пройдены")

проверки пройдены


## Шаг 23. Сохраняем очищенную таблицу

Сырые файлы не трогаем: результат пишем в `data/processed/`. Время в формате `+0000`. Файл перезаписывается при каждом запуске. Читаем его обратно и проверяем ещё раз: то, что лежит на диске, тоже должно проходить контракт.

Перед записью проверяем, что период погоды накрывает все часы станции: это нужно для объединения в задаче 3.1.

In [26]:
# погода должна накрывать все часы станции
print("погода:", weather["time_utc"].min(), "…", weather["time_utc"].max())
print("станция:", pm25_clean["time_utc"].min(), "…", pm25_clean["time_utc"].max())
assert weather["time_utc"].min() <= pm25_clean["time_utc"].min(), "погода начинается позже станции"
assert weather["time_utc"].max() >= pm25_clean["time_utc"].max(), "погода кончается раньше станции"

clean.save_clean_csv(pm25_clean, PM25_CLEAN_PATH)

pm25_read = clean.read_clean_csv(PM25_CLEAN_PATH)
clean.check_clean_pm25(pm25_read)
print("pm25_clean:", pm25_read.shape)
print("первая метка:", pm25_read["time_utc"].min(), "последняя:", pm25_read["time_utc"].max())
print("pm25_missing True:", pm25_read["pm25_missing"].sum())
print("pm25_interpolated True:", pm25_read["pm25_interpolated"].sum())

with open(PM25_CLEAN_PATH) as file:
    for _ in range(4):
        print(file.readline().rstrip())

погода: 2020-04-09 00:00:00+00:00 … 2025-11-14 23:00:00+00:00
станция: 2020-04-09 16:00:00+00:00 … 2025-11-14 10:00:00+00:00


pm25_clean: (49075, 4)
первая метка: 2020-04-09 16:00:00+00:00 последняя: 2025-11-14 10:00:00+00:00
pm25_missing True: 10235
pm25_interpolated True: 1148
time_utc,pm25,pm25_missing,pm25_interpolated
2020-04-09T16:00:00+0000,6.0,False,False
2020-04-09T17:00:00+0000,10.0,False,False
2020-04-09T18:00:00+0000,8.0,False,False


## Итог части 2

**Что нашли и что изменили** (числа — в выводе шагов 12–23):

- Значения с флагом поставщика заменили пропусками.
- Застывший датчик (одно значение 6 часов подряд и дольше) заменили пропусками.
- Нули: решение принято по выводу шага 19, одиночные нули оставили.
- Выбросы по IQR и пики смога не удаляли: это изучаемое явление.
- Короткие дыры (до 3 часов целиком) заполнили интерполяцией и пометили `pm25_interpolated`.
- Длинные дыры оставили пустыми и пометили `pm25_missing`.

**Файл** `data/processed/pm25_clean.csv`: колонки `time_utc, pm25, pm25_missing, pm25_interpolated`, полная часовая сетка станции (число строк и границы — в выводе шага 23).

**Для задачи 3.1:**

- Ключ `time_utc` — начало часа измерения.
- Присоединять погоду так: `pm25.merge(weather, on="time_utc", how="left", validate="one_to_one")`. Строк остаётся столько же, сколько в сетке станции, пустой погоды нет. Погода шире периода станции и накрывает все её часы (границы обеих таблиц — в выводе шага 23).
- CAMS до 2022-08-04 пуст: источник ещё не существовал (см. часть 1).
- Решить в 3.1, считать ли часы с `pm25_interpolated` в правиле «не менее 18 часов из 24» для суточного среднего.